# Merge Dataset Scraping ABSA Hotel Santika

Notebook ini **HANYA melakukan merge** data scraping dari tiga platform
(**Agoda, Tiket, Traveloka**) menjadi satu dataset mentah terpadu. Tidak ada
cleaning, labeling, atau EDA di sini.

**Termasuk Agoda Santika Bekasi** (tidak dikecualikan).

## Output
`dataset_absa_santika_merged_raw.csv` dengan kolom:
`ID_Review, Platform, Nama_Hotel, Review_Date, Text_Review`

## Cara pakai di Kaggle
1. Upload folder `Agoda`, `Tiket`, `Traveloka` (berisi CSV per hotel) sebagai dataset input.
2. Run All. Notebook mencari ketiga folder secara otomatis di `/kaggle/input`.

## 1. Import dan Konfigurasi

In [ ]:
from pathlib import Path
import re
from datetime import datetime, timedelta
import pandas as pd

# Tanggal scraping Traveloka (untuk konversi 'Diulas X minggu lalu').
# SESUAIKAN bila perlu dengan tanggal scraping sebenarnya.
TRAVELOKA_SCRAPE_DATE = datetime(2026, 6, 6)

PLATFORMS = ['Agoda', 'Tiket', 'Traveloka']
OUTPUT_FILE = 'dataset_absa_santika_merged_raw.csv'
print('Platforms:', PLATFORMS)

## 2. Menemukan Folder Platform

Mendukung Kaggle (`/kaggle/input/...`) maupun lokal. Folder yang dicari adalah
yang memuat sub-folder `Agoda`, `Tiket`, `Traveloka`.

In [ ]:
def has_platform_dirs(path):
    path = Path(path)
    return all((path / p).exists() for p in PLATFORMS)

def resolve_scraping_dir():
    candidates = [Path.cwd(), Path.cwd() / 'Data Scraping',
                  Path('Data Scraping')]
    ki = Path('/kaggle/input')
    if ki.exists():
        for root in ki.iterdir():
            candidates += [root, root / 'Data Scraping']
            # kadang folder platform langsung di dalam dataset
    for c in candidates:
        if has_platform_dirs(c):
            return Path(c)
    raise FileNotFoundError(
        'Folder berisi Agoda, Tiket, Traveloka tidak ditemukan. '
        'Upload ketiganya sebagai input Kaggle.')

SCRAPING_DIR = resolve_scraping_dir()
print('Scraping source:', SCRAPING_DIR)
for p in PLATFORMS:
    files = list((SCRAPING_DIR / p).glob('*.csv'))
    print(f'  {p}: {len(files)} CSV')

## 3. Helper: Nama Hotel Kanonik dan Parsing Tanggal

Nama hotel diturunkan dari nama berkas dan dinormalisasi ke bentuk kanonik.
Tanggal dinormalisasi ke ISO `YYYY-MM-DD` sesuai format tiap platform.

In [ ]:
def normalize_hotel_name(filename):
    name = re.sub(r'\.csv$', '', filename, flags=re.IGNORECASE)
    n = name.lower()
    if 'megacity' in n or 'mega city' in n or 'bekasi' in n:
        return 'Hotel Santika Megacity Bekasi'
    if 'pasir koja' in n or 'pasirkoja' in n:
        return 'Hotel Santika Bandung - Jalan Pasir Koja'
    if 'pasir kaliki' in n or 'pasirkaliki' in n:
        return 'Hotel Santika Bandung - Jalan Pasir Kaliki'
    if 'sumatera' in n:
        return 'Hotel Santika Bandung - Jalan Sumatera'
    if 'bandung' in n:
        return 'Hotel Santika Bandung'
    if 'bogor' in n:
        return 'Hotel Santika Bogor'
    if 'cirebon' in n:
        return 'Hotel Santika Cirebon'
    if 'depok' in n:
        return 'Hotel Santika Depok'
    if 'tasik' in n:
        return 'Hotel Santika Tasikmalaya'
    return 'Hotel Santika ' + name.strip()

BULAN_EN = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,'july':7,
            'august':8,'september':9,'october':10,'november':11,'december':12,
            'jan':1,'feb':2,'mar':3,'apr':4,'jun':6,'jul':7,'aug':8,'sep':9,'sept':9,
            'oct':10,'nov':11,'dec':12}

def to_iso_generic(val):
    v = str(val).strip()
    if v == '':
        return ''
    if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
        return v
    # Inggris month-first: May 17, 2026
    m = re.match(r'^([A-Za-z]+)\s+(\d{1,2}),\s*(\d{4})$', v)
    if m and m.group(1).lower() in BULAN_EN:
        return f'{int(m.group(3)):04d}-{BULAN_EN[m.group(1).lower()]:02d}-{int(m.group(2)):02d}'
    # day-first: 19 May 2026 / 28 Apr 2026
    m = re.match(r'^(\d{1,2})\s+([A-Za-z]+)\.?\s+(\d{2,4})$', v)
    if m and m.group(2).lower() in BULAN_EN:
        y = int(m.group(3)); y = y + 2000 if y < 100 else y
        return f'{y:04d}-{BULAN_EN[m.group(2).lower()]:02d}-{int(m.group(1)):02d}'
    # dash short: 16-Apr-26
    m = re.match(r'^(\d{1,2})-([A-Za-z]{3,})-(\d{2,4})$', v)
    if m and m.group(2).lower() in BULAN_EN:
        y = int(m.group(3)); y = y + 2000 if y < 100 else y
        return f'{y:04d}-{BULAN_EN[m.group(2).lower()]:02d}-{int(m.group(1)):02d}'
    return ''  # gagal -> dibiarkan kosong (merge tetap jalan)

def traveloka_relative_to_iso(val, scrape_date):
    v = str(val).strip().lower()
    m = re.search(r'diulas\s+(\d+)\s+minggu\s+lalu', v)
    if m:
        return (scrape_date - timedelta(weeks=int(m.group(1)))).strftime('%Y-%m-%d')
    m = re.search(r'diulas\s+(\d+)\s+hari\s+lalu', v)
    if m:
        return (scrape_date - timedelta(days=int(m.group(1)))).strftime('%Y-%m-%d')
    m = re.search(r'diulas\s+(\d+)\s+bulan\s+lalu', v)
    if m:
        return (scrape_date - timedelta(days=int(m.group(1)) * 30)).strftime('%Y-%m-%d')
    # kalau sudah ISO atau format lain, coba generic
    return to_iso_generic(val)

def find_col(df, *names):
    lut = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lut:
            return lut[n.lower()]
    return None

print('Helper siap.')

## 4. Membaca dan Menggabungkan Semua Platform

Tiap platform punya skema kolom berbeda, namun semua memiliki `teks_ulasan`
dan `tanggal`. Kita ambil teks + tanggal, tambahkan Platform & Nama_Hotel.

In [ ]:
rows = []
for platform in PLATFORMS:
    folder = SCRAPING_DIR / platform
    for f in sorted(folder.glob('*.csv')):
        hotel = normalize_hotel_name(f.name)
        try:
            df = pd.read_csv(f, dtype=str, keep_default_na=False, encoding='utf-8-sig')
        except Exception:
            df = pd.read_csv(f, dtype=str, keep_default_na=False, encoding='utf-8')
        text_col = find_col(df, 'teks_ulasan', 'text_review', 'review', 'text')
        date_col = find_col(df, 'tanggal', 'review_date', 'date')
        if text_col is None:
            print(f'  [SKIP] {platform}/{f.name}: kolom teks tidak ditemukan')
            continue
        for _, r in df.iterrows():
            text = str(r.get(text_col, '')).strip()
            raw_date = str(r.get(date_col, '')).strip() if date_col else ''
            if platform == 'Traveloka':
                iso = traveloka_relative_to_iso(raw_date, TRAVELOKA_SCRAPE_DATE)
            else:
                iso = to_iso_generic(raw_date)
            rows.append({
                'Platform': platform,
                'Nama_Hotel': hotel,
                'Review_Date': iso,
                'Text_Review': text,
            })
        print(f'  {platform}/{f.name}: {len(df)} baris -> {hotel}')

merged = pd.DataFrame(rows)
print('\nTotal baris hasil merge:', len(merged))

## 5. Menambahkan ID Review dan Menyusun Kolom Final

In [ ]:
merged = merged.reset_index(drop=True)
merged.insert(0, 'ID_Review', ['REV-{:06d}'.format(i + 1) for i in range(len(merged))])
merged = merged[['ID_Review', 'Platform', 'Nama_Hotel', 'Review_Date', 'Text_Review']]

print('Distribusi platform:')
print(merged['Platform'].value_counts().to_string())
print('\nDistribusi hotel:')
print(merged['Nama_Hotel'].value_counts().to_string())
print('\nTanggal ISO valid:', int(merged['Review_Date'].str.match(r'^\d{4}-\d{2}-\d{2}$').sum()), '/', len(merged))
merged.head()

## 6. Menyimpan Dataset Merge

In [ ]:
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
out_path = out_dir / OUTPUT_FILE
merged.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Tersimpan:', out_path)
print('Total baris:', len(merged))